# Glimmer Q4 on Kaggle (rung 3)

**PRIVACY: this Kaggle rung is for PUBLIC data only** (judgment extraction, evaluation). Never send it consented documents (runbook 15) or user uploads; spec section 6: Kaggle is never shown user documents.

Settings: Accelerator = GPU T4 x2, Internet = On. Check your weekly GPU quota (Profile > Settings, or the quota meter at the top right) before running; a session burns quota while it is open. Tunnel tool: cloudflared quick tunnel (free, no account).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, secrets
KEY = secrets.token_urlsafe(24)  # the API key for this session; paste it into KAGGLE_KEY locally
MODEL_DIR = '/kaggle/tmp/glimmer'
os.makedirs(MODEL_DIR, exist_ok=True)
print('Do NOT publish or share this notebook with saved outputs: they contain the key and tunnel URL.')
print('KEY (copy to your local .env as KAGGLE_KEY):', KEY)

In [ ]:
# Disk check: Q4 (~17 GB) + mmproj (3.6 GB) + llama.cpp build need about 25 GB. /kaggle/working is only ~20 GB, so everything goes in /kaggle/tmp.
import os, shutil
os.makedirs('/kaggle/tmp', exist_ok=True)
free_gb = shutil.disk_usage('/kaggle/tmp').free / 1e9
print(f'free in /kaggle/tmp: {free_gb:.1f} GB')
!df -h /kaggle/tmp
DISK_OK = free_gb >= 25
if not DISK_OK:
    print(f'STOP: only {free_gb:.1f} GB free in /kaggle/tmp, need at least 25 GB. Later cells will refuse to run.')

In [ ]:
assert DISK_OK, "Disk check failed or was not run: stop here. Free space in /kaggle/tmp or use the HPC/Baseten rungs."
# Download the UD-Q4_K_XL GGUF and mmproj with curl (HF repo/path: fill in the same source used for the cluster files).
HF_REPO = 'REPLACE_WITH_GLIMMER_HF_REPO'  # placeholder: the repo you downloaded the cluster weights from
!curl -L -C - -o {MODEL_DIR}/Muse-Glimmer-30B-UD-Q4_K_XL.gguf https://huggingface.co/{HF_REPO}/resolve/main/Muse-Glimmer-30B-UD-Q4_K_XL.gguf
!curl -L -C - -o {MODEL_DIR}/mmproj-Muse-Glimmer-30B-BF16.gguf https://huggingface.co/{HF_REPO}/resolve/main/mmproj-Muse-Glimmer-30B-BF16.gguf
!ls -la {MODEL_DIR}

In [ ]:
assert DISK_OK, "Disk check failed or was not run: stop here. Free space in /kaggle/tmp or use the HPC/Baseten rungs."
# Build llama-server with CUDA for T4 (compute capability 7.5). Takes about 10 to 15 minutes.
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /kaggle/tmp/llama.cpp
!cmake -S /kaggle/tmp/llama.cpp -B /kaggle/tmp/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF > /kaggle/tmp/build.log 2>&1
!cmake --build /kaggle/tmp/llama.cpp/build --target llama-server -j 4 >> /kaggle/tmp/build.log 2>&1; tail -3 /kaggle/tmp/build.log

In [ ]:
assert DISK_OK, "Disk check failed or was not run: stop here. Free space in /kaggle/tmp or use the HPC/Baseten rungs."
# Start llama-server split across the 2 T4s.
import subprocess
kf = '/kaggle/tmp/api_key'
fd = os.open(kf, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
os.write(fd, KEY.encode()); os.close(fd)  # key file, 0600, so the key is not in the process list
cmd = ['/kaggle/tmp/llama.cpp/build/bin/llama-server','-m',f'{MODEL_DIR}/Muse-Glimmer-30B-UD-Q4_K_XL.gguf','--mmproj',f'{MODEL_DIR}/mmproj-Muse-Glimmer-30B-BF16.gguf','--alias','glimmer-q4','-ngl','99','--tensor-split','1,1','-c','16384','--jinja','--host','127.0.0.1','--port','18080','--api-key-file','/kaggle/tmp/api_key']
srv = subprocess.Popen(cmd, stdout=open('/kaggle/tmp/server.log','w'), stderr=subprocess.STDOUT)
import time; time.sleep(90); print(open('/kaggle/tmp/server.log').read()[-800:])

In [ ]:
assert DISK_OK, "Disk check failed or was not run: stop here. Free space in /kaggle/tmp or use the HPC/Baseten rungs."
# Expose through a cloudflared quick tunnel (free, no account). The server still requires the API key.
!curl -L -o /kaggle/tmp/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /kaggle/tmp/cloudflared
import re
tun = subprocess.Popen(['/kaggle/tmp/cloudflared','tunnel','--url','http://127.0.0.1:18080'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tun.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m: url = m.group(0); break
print('KAGGLE_BASE_URL =', url + '/v1')

In [ ]:
assert DISK_OK, "Disk check failed or was not run: stop here. Free space in /kaggle/tmp or use the HPC/Baseten rungs."
# Keep the session alive while you work. Stop the notebook session when finished to save GPU quota.
import time
while True: time.sleep(600)